In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

df = pd.read_csv("../data/2020-Apr-L.csv")

In [4]:
# PREPARACIÓN
df["cat_l2"] = df["category_code"].apply(lambda x: ".".join(x.split(".")[:2]))
df["is_unknown"] = df["brand"] == "unknown"

In [5]:
# MÉTRICA GLOBAL
pct_unknown_global = df["is_unknown"].mean() * 100
print(f"=== MÉTRICA CLAVE ===")
print(f"%_unknown_global : {pct_unknown_global:.2f}%")
if pct_unknown_global > 30:
    print("Supera el 30% → documentar limitación en informe")
elif pct_unknown_global > 20:
    print("Supera el 20% → cobertura limitada para modelo de contenido")
else:
    print("Bajo el 20% → impacto manejable")

=== MÉTRICA CLAVE ===
%_unknown_global : 25.84%
Supera el 20% → cobertura limitada para modelo de contenido


In [8]:
# 3. AGREGACIÓN POR SUBCATEGORÍA (nivel 2)
# ─────────────────────────────────────────────
grp = df.groupby("cat_l2")
 
freq_total   = grp.size().rename("total")
freq_unknown = grp["is_unknown"].sum().rename("unknown")
freq_known   = (freq_total - freq_unknown).rename("known")
pct_unknown  = (freq_unknown / freq_total * 100).rename("pct_unknown")
 
# Tasa de conversión: purchase / total eventos por grupo y brand_type
def conv_rate(mask):
    sub = df[mask]
    return (sub["event_type"] == "purchase").sum() / len(sub) * 100 if len(sub) > 0 else 0
 
conv_unknown = grp.apply(lambda g: (
    (g[g["is_unknown"]]["event_type"] == "purchase").sum() / len(g[g["is_unknown"]]) * 100
    if len(g[g["is_unknown"]]) > 0 else 0.0
)).rename("conv_unknown")

conv_known = grp.apply(lambda g: (
    (g[~g["is_unknown"]]["event_type"] == "purchase").sum() / len(g[~g["is_unknown"]]) * 100
    if len(g[~g["is_unknown"]]) > 0 else 0.0
)).rename("conv_known")
 
resumen = pd.concat([freq_total, freq_unknown, freq_known,
                     pct_unknown, conv_unknown, conv_known], axis=1)
 
# Filtro de ruido: excluir subcategorías con <0.5% del total
total_eventos = len(df)
resumen = resumen[freq_total / total_eventos * 100 >= 0.5]
 
# Ordenar por % unknown descendente
resumen = resumen.sort_values("pct_unknown", ascending=True)
 
print(f"\n=== POR SUBCATEGORÍA ===")
print(resumen[["total", "pct_unknown", "conv_unknown", "conv_known"]].to_string())
 
# ─────────────────────────────────────────────
# 4. FIGURA: 2 paneles
#    Panel A → Barras horizontales apiladas
#    Panel B → Tasa de conversión Unknown vs Known
# ─────────────────────────────────────────────
n_cats = len(resumen)
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(15, max(6, n_cats * 0.55)),
    gridspec_kw={"width_ratios": [2.5, 1], "wspace": 0.05}
)
fig.patch.set_facecolor("#F8FAFC")
ax1.set_facecolor("#F8FAFC")
ax2.set_facecolor("#F8FAFC")
 
# Labels del eje Y (limpiar prefijos)
y_labels = [c.replace("apparel.", "app.")
             .replace("accessories.", "acc.")
             .replace("sport.", "sport.")
            for c in resumen.index]
y_pos = np.arange(n_cats)
 
# ── Panel A: Barras apiladas ──────────────────
pct_unk = resumen["pct_unknown"].values
pct_kno = 100 - pct_unk
 
bars_known = ax1.barh(y_pos, pct_kno, color="#60A5FA",
                      height=0.6, label="Marcas Reales", zorder=3)
bars_unk   = ax1.barh(y_pos, pct_unk, left=pct_kno,
                      color="#F87171", height=0.6, label="Unknown", zorder=3)
 
# Etiqueta % unknown dentro de la barra si hay espacio
for i, (pu, pk) in enumerate(zip(pct_unk, pct_kno)):
    if pu >= 5:
        ax1.text(pk + pu / 2, i, f"{pu:.1f}%",
                 ha="center", va="center",
                 fontsize=7.5, color="white", fontweight="bold")
 
# Línea de referencia 20% y 30%
for thresh, color, label in [(20, "#D97706", "20%"), (30, "#DC2626", "30%")]:
    ax1.axvline(thresh, color=color, linewidth=1.2,
                linestyle="--", zorder=4)
    ax1.text(thresh + 0.5, n_cats - 0.3, label,
             fontsize=7.5, color=color, fontweight="bold")
 
# Cuadrícula vertical suave
for x in [25, 50, 75, 100]:
    ax1.axvline(x, color="#E2E8F0", linewidth=0.7, zorder=1)
 
ax1.set_xlim(0, 105)
ax1.set_xticks([0, 25, 50, 75, 100])
ax1.set_xticklabels(["0%", "25%", "50%", "75%", "100%"], fontsize=8.5)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(y_labels, fontsize=9)
ax1.set_xlabel("Proporción de eventos (%)", fontsize=9, color="#334155", labelpad=8)
ax1.tick_params(axis="both", length=0)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["left"].set_color("#E2E8F0")
ax1.spines["bottom"].set_color("#E2E8F0")
 
# Leyenda panel A
legend_patches = [
    Patch(color="#60A5FA", label="Marcas Reales"),
    Patch(color="#F87171", label='"Unknown" (imputado)'),
]
ax1.legend(handles=legend_patches, fontsize=8.5,
           frameon=False, loc="lower right")
 
# ── Panel B: Conversión Unknown vs Known ──────
ax2.set_facecolor("#F8FAFC")
conv_unk_vals = resumen["conv_unknown"].values
conv_kno_vals = resumen["conv_known"].values
 
ax2.barh(y_pos - 0.18, conv_kno_vals, height=0.33,
         color="#60A5FA", label="Marcas Reales", zorder=3)
ax2.barh(y_pos + 0.18, conv_unk_vals, height=0.33,
         color="#F87171", label='"Unknown"', zorder=3)
 
# Etiquetas de valor
for i, (cu, ck) in enumerate(zip(conv_unk_vals, conv_kno_vals)):
    ax2.text(max(cu, 0.05) + 0.02, i + 0.18,
             f"{cu:.2f}%", va="center", fontsize=6.5, color="#7F1D1D")
    ax2.text(max(ck, 0.05) + 0.02, i - 0.18,
             f"{ck:.2f}%", va="center", fontsize=6.5, color="#1E40AF")
 
ax2.set_yticks(y_pos)
ax2.set_yticklabels([])
ax2.set_xlabel("Conv. Rate (%)", fontsize=9, color="#334155", labelpad=8)
ax2.tick_params(axis="both", length=0)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["bottom"].set_color("#E2E8F0")
 
for x in ax2.get_xticks():
    ax2.axvline(x, color="#E2E8F0", linewidth=0.7, zorder=1)
 
ax2.legend(fontsize=7.5, frameon=False, loc="lower right")
 
# ── Títulos ───────────────────────────────────
fig.suptitle('Impacto de la Imputación en brand ("Unknown")',
             fontsize=13, fontweight="bold", x=0.0,
             ha="left", y=1.02, color="#0F172A",
             transform=ax1.transAxes)
ax1.text(0.0, 1.025,
         f"Proporción Unknown vs Marcas Reales por subcategoría · %_unknown_global = {pct_unknown_global:.1f}%  "
         f"{'⚠️ Supera umbral crítico (30%)' if pct_unknown_global > 30 else ''}",
         transform=ax1.transAxes, fontsize=8.5, color="#64748B")
 
ax2.set_title("Validación\nCruzada", fontsize=9,
              fontweight="bold", color="#334155", pad=8)
 
plt.tight_layout()
plt.savefig("../eda_outputs/06_brand_unknown_impact.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print("¡Gráfico 06_brand_unknown_impact.png guardado exitosamente!")


=== POR SUBCATEGORÍA ===
                      total  pct_unknown  conv_unknown  conv_known
cat_l2                                                            
apparel.trousers     207533     6.305021      0.573175    2.636695
apparel.jeans        111188     7.390186      0.949252    0.740014
apparel.costume      453153    14.618904      0.695891    0.787786
accessories.wallet    99323    16.367810      0.787353    0.591096
apparel.shoes       4585989    18.423441      0.954199    1.159554
apparel.scarf        508156    24.967136      1.310770    0.894609
accessories.bag      621076    25.990217      0.776241    0.922427
apparel.jumper        52487    27.239126      0.734420    0.573449
sport.trainer        885813    28.260592      0.717436    0.834649
apparel.underwear    492767    36.365057      0.441419    0.899315
apparel.sock          89434    36.639309      0.793457    0.610595
apparel.tshirt       204489    38.101316      0.605804    0.981229
apparel.shirt        538396    42.35

C:\Users\Juan\AppData\Local\Temp\ipykernel_12340\3735338125.py:151: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


¡Gráfico 06_brand_unknown_impact.png guardado exitosamente!
